In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [ ]:
from transformers import AutoProcessor

# model_name_or_path = "/projects/bhuang/models/llm/pretrained/google/gemma-3n-E4B-it"
# model_name_or_path = "/home/bhuang/llm/trl/outputs/intent_classification/audio_ft/sga/sft_gemma3n_e4b_it_lora_r64_ep1_bs64_lr2e4_merged"
model_name_or_path = "/home/bhuang/llm/trl/outputs/intent_classification/audio_ft/sga/sft_gemma3n_e4b_it_lora_r64_ep5_bs64_lr2e4_merged"

processor = AutoProcessor.from_pretrained(model_name_or_path)

type(processor)

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
).eval()

type(model)

In [ ]:
# audio_filepath = "/home/bhuang/llm/momo/intent_classification/data/sga/audio_degraded/000000.wav"
audio_filepath = "/home/bhuang/llm/momo/intent_classification/data/databank/audio_degraded/train/000000.wav"

messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are an assistant that transcribes speech accurately.",
            }
        ],
    },
    {
        "role": "user",
        "content": [
            # voxtral
            # {"type": "audio", "url": audio_url},
            # {"type": "audio", "path": audio_filepath},
            # {"type": "audio", "base64": audio_base64},
            # gemma-3n, qwen-omni
            {"type": "audio", "audio": audio_filepath},
            {"type": "text", "text": "Please transcribe this audio."},
        ],
    },
    {
        "role": "assistant",
        "content": [
            {"type": "text", "text": "Hello! How are you? fine thank you"},
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=False,
    # add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
list(inputs.keys()), inputs

In [ ]:
with torch.inference_mode():
    outputs = model.generate(
        **inputs.to(model.device, dtype=model.dtype),
        max_new_tokens=100,
        # do_sample=False,
        # disable_compile=True,
    )

decoded_outputs = processor.batch_decode(
    outputs[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
    # clean_up_tokenization_spaces=True,
)

print(decoded_outputs[0])